In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd

import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


In [3]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
submission_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# Q1) Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [5]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
most_frequent_option_count=train_df['answer'].value_counts().max()
min_frequent_option_count=train_df['answer'].value_counts().min()
sum=most_frequent_option_count+min_frequent_option_count
print(f"{most_frequent_option_count}+ {min_frequent_option_count}={sum}")

490+ 324=814


# Q2) After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [7]:
words=train_df['prompt'].dropna().astype(str).str.lower().str.translate(str.maketrans('', '', string.punctuation)).str.split().explode()
words.nunique()

859

# Q3) Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [8]:
prompt = train_df['prompt'].iloc[0]

cleaned_prompt = prompt.lower().translate(str.maketrans('', '', string.punctuation))


words = cleaned_prompt.split()
print(words)

filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]
print(filtered_words)

len(filtered_words)

['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


13

# Q4)Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [9]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
all_columns=['prompt','A','B','C','D','E']
combined_text_of_all_columns = train_df[all_columns].fillna('').astype(str).agg(' '.join, axis=1)
print(combined_text_of_all_columns.iloc[0])

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. Martin Heidegger believes that the relationship between time and hu

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_of_all_columns)

vocabulary_size = len(vectorizer.vocabulary_)
print(f"Vocabulary Size:{vocabulary_size}")

Vocabulary Size:2762


# Q5)Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_text = train_df.iloc[0,1]
option_a_text = train_df.iloc[0,2]

prompt_vector = vectorizer.transform([prompt_text])
option_a_vector = vectorizer.transform([option_a_text])

similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]
round(similarity_score, 4)

np.float64(0.272)

# Q6)Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [13]:
all_columns=['prompt','A','B','C','D','E']
combined_text_of_all_columns = train_df[all_columns].fillna('').astype(str).agg(' '.join, axis=1)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_of_all_columns)

TfidfVectorizer(stop_words='english')

In [14]:
correct_predictions = 0
total_rows = len(train_df)

for index, row in train_df.iterrows():
    prompt_text = row['prompt']
    actual_answer_text = row['answer']
    
    prompt_vector = vectorizer.transform([prompt_text])
    
    # Calculate similarity for each of the 5 options
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_text = row[option]
        option_vector = vectorizer.transform([option_text])
        
        score = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities[option] = score
        
    best_option = max(similarities, key=similarities.get)
    
    if best_option == actual_answer_text:
        correct_predictions += 1

accuracy_percentage = (correct_predictions / total_rows) * 100
accuracy_percentage

13.55

# Q7)If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [15]:
truth_answer = 'C'
prediction = ['C', 'A', 'B']

if truth_answer == prediction[0]:
    score = 1.0
elif truth_answer == prediction[1]:
    score = 0.5
elif truth_answer == prediction[2]:
    score = 1/3
else:
    score = 0.0

print(f"MAP@3 Score: {score:.4f}")

MAP@3 Score: 1.0000


# Q8)If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

In [16]:
truth_answer = 'B'
prediction = ['D', 'B', 'E']

if truth_answer == prediction[0]:
    score = 1.0
elif truth_answer == prediction[1]:
    score = 0.5
elif truth_answer == prediction[2]:
    score = 1/3
else:
    score = 0.0

print(f"MAP@3 Score: {score:.4f}")

MAP@3 Score: 0.5000


# Q9)The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [17]:
top_3 = train_df['answer'].value_counts().index[:3].tolist()

scores = []
for correct_answer in train_df['answer']:
    if correct_answer == top_3[0]:
        scores.append(1.0)
    elif correct_answer == top_3[1]:
        scores.append(0.5)
    elif correct_answer == top_3[2]:
        scores.append(1/3)
    else:
        scores.append(0.0)

overall_map3 = np.sum(scores) / len(scores)
overall_map3

np.float64(0.42125)

# Q10)The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [18]:
all_columns=['prompt','A','B','C','D','E']
combined_text_of_all_columns = train_df[all_columns].fillna('').astype(str).agg(' '.join, axis=1)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_of_all_columns)

TfidfVectorizer(stop_words='english')

In [19]:
scores = []

for index, row in train_df.iterrows():
    prompt_text = row['prompt']
    actual_answer_text = row['answer']
    
    prompt_vector = vectorizer.transform([prompt_text])
    
    # Calculate similarity for each of the 5 options
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_text = row[option]
        option_vector = vectorizer.transform([option_text])
        
        score = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities[option] = score
        
    sorted_options = sorted(similarities, key=similarities.get, reverse=True)
    top_3 = sorted_options[:3]
    
    if actual_answer_text == top_3[0]:
        scores.append(1.0)
    elif actual_answer_text == top_3[1]:
        scores.append(0.5)
    elif actual_answer_text == top_3[2]:
        scores.append(1/3)
    else:
        scores.append(0.0)

final_map3 = np.mean(scores)
round(final_map3 ,4)

np.float64(0.2962)